In [37]:
import pandas as pd
import seaborn as sns
import xgboost as xgb
import numpy as np

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neural_network import MLPClassifier

from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


In [38]:
"""
---------------------------------------------------------------------------------------------------------------
Implementação de ensemble baseado em Blending dos modelos EVI, MMAING e EARS com um meta-learner MLP.
---------------------------------------------------------------------------------------------------------------
Metodologia: 
O Stacking, ou generalização empilhada, é uma técnica de ensemble que combina múltiplos modelos para gerar uma nova predição. 
A estratégia é usar as predições de vários modelos base (chamados de "nível 0") como entrada para um novo modelo, conhecido como meta-modelo ou blender ("nível 1"). 
Este blender aprende a melhor forma de combinar as predições dos modelos base para obter um resultado final mais acurado.

Modelos Base (Nível 0): EARS, EVI, ......

Meta-Modelo (Nível 1): MLPClassifier.

---------------------------------------------------------------------------------------------------------------
Princípios de Funcionamento:
O MLPClassifier (Multi-Layer Perceptron) é uma rede neural artificial feedforward. 
Diferente de algoritmos tradicionais baseados em árvores de decisão, o MLP é capaz de mapear relações matemáticas complexas e não-lineares entre as variáveis de entrada e a variável alvo.
---------------------------------------------------------------------------------------------------------------
Autor:
Andrêza Leite de Alencar, PhD
Federal Rural University of Pernambuco (UFRPE)
Center for Data and Knowledge Integration for Health (CIDACS/FIOCRUZ)

---------------------------------------------------------------------------------------------------------------
Data: 
10/11/2025

---------------------------------------------------------------------------------------------------------------

"""

'\n---------------------------------------------------------------------------------------------------------------\nImplementação de ensemble baseado em Blending dos modelos EVI, MMAING e EARS com um meta-learner MLP.\n---------------------------------------------------------------------------------------------------------------\nMetodologia: \nO Stacking, ou generalização empilhada, é uma técnica de ensemble que combina múltiplos modelos para gerar uma nova predição. \nA estratégia é usar as predições de vários modelos base (chamados de "nível 0") como entrada para um novo modelo, conhecido como meta-modelo ou blender ("nível 1"). \nEste blender aprende a melhor forma de combinar as predições dos modelos base para obter um resultado final mais acurado.\n\nModelos Base (Nível 0): EARS, EVI, ......\n\nMeta-Modelo (Nível 1): MLPClassifier.\n\n---------------------------------------------------------------------------------------------------------------\nPrincípios de Funcionamento:\nO ML

# --- 1. Carregamento dos Dados ---

In [39]:
try:
    #df = pd.read_parquet('data_ens_1_11_25.parquet')
    #df = pd.read_parquet('dado.parquet')
    #df = pd.read_parquet('resultado_xgb_31_03_2026.parquet')
    df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/resultado_predicoes_producao_xgb.parquet')

except FileNotFoundError:
    print("Arquivo não encontrado.")
    
    df = pd.DataFrame(data)

# --- 2. Preparação dos Dados ---


In [40]:
#=====================================================================
#!!!!!!!  ajustar com o dado final !!!!!!! 
#=====================================================================

columns_sinais = [
    #'farrrignton', #não esta no arquivo
    'sinal_ears_atend',
     'sinal_evi_ivas',
     'EWS_ISF',
     'EWS_LOF',
     'EWS_OCSVM',
     'EWS_COPOD',
     'EWS_Rt' #NGM
] 

## --- 2.1 Tratamento de NANs ---

In [41]:
for col in columns_sinais:
    if df[col].isnull().any():
        moda = df[col].mode()[0]
        df[col].fillna(moda, inplace=True)
        print(f"Valores NaN na coluna '{col}' preenchidos com a moda: {int(moda)}")

print("\nVerificando dados faltantes (NaN) após o tratamento:")
print(df.isnull().sum())


Verificando dados faltantes (NaN) após o tratamento:
co_ibge                             0
epiweek                             0
year                                0
year_week                           0
week                                0
atend_ivas                          0
atend_totais                        0
mem_surge_01_correct_with_consec    0
warning_final_mem_surge_01          0
sinal_ears_atend                    0
sinal_evi_ivas                      0
EWS_ISF                             0
EWS_LOF                             0
EWS_OCSVM                           0
EWS_COPOD                           0
EWS_Rt                              0
prob_ensemble_xgb                   0
signal_ensemble_xgb50               0
dtype: int64


## --- 2.2 Função para criar Lag Features (Janelas de Atraso) ---


In [42]:
# --- Função para criar Lag Features (Janelas de Atraso) ---
def criar_lag_features(df, colunas_sinais, lags=[1, 2, 3]):
    """
    Cria colunas de atraso (lags) para dar contexto histórico ao modelo.
    Ex: Se lag=1, cria uma coluna com o valor da semana passada.
    """
    df_lagged = df.copy()
    
    # Ordena por tempo ('year_week') e local ('co_ibge7') para garantir que o shift pegue a semana anterior correta
    if 'co_ibge7' in df.columns:
        df_lagged = df_lagged.sort_values(by=['co_ibge7', 'year_week'])
    else:
        df_lagged = df_lagged.sort_values(by=['year_week'])

    new_features = []
    
    for col in colunas_sinais:
        for lag in lags:
            nome_nova_coluna = f"{col}_lag{lag}"
            # O shift(lag) empurra os valores para baixo
            if 'co_ibge7' in df.columns:
                # Agrupa por município para não pegar dado de uma cidade e jogar na outra
                df_lagged[nome_nova_coluna] = df_lagged.groupby('co_ibge7')[col].shift(lag)
            else:
                df_lagged[nome_nova_coluna] = df_lagged[col].shift(lag)
            
            new_features.append(nome_nova_coluna)
    
    # Trata as primeiras linhas que ficaram vazias (NaN) por causa do shift
    for col in new_features:
        if df_lagged[col].isnull().any():
            moda = df_lagged[col].mode()[0] # .mode() retorna uma série, pegamos o primeiro valor
            df_lagged[col].fillna(0, inplace=True)
            print(f"Valores NaN na coluna '{col}' preenchidos com a moda: {int(moda)}")
    #df_lagged = df_lagged.dropna()

    
    return df_lagged, new_features



## --- 2.2.1 Aplicação da função no DataFrame ---


In [43]:
# --- Aplicação no DataFrame ---

# Cria os lags de 1, 2 e 3 semanas atrás
df_com_lags, nomes_features_lags = criar_lag_features(df, columns_sinais, lags=[1, 2, 3])

print(f"Novas features criadas: {nomes_features_lags}")
print(f"Tamanho do dataset original: {len(df)}")
print(f"Tamanho do dataset com lags: {len(df_com_lags)}")

# ---  Atualiza as variáveis de Treino ---
# Agora a lista de features deve incluir as originais E as novas
features_para_treino = columns_sinais + nomes_features_lags


Valores NaN na coluna 'sinal_ears_atend_lag1' preenchidos com a moda: 0
Valores NaN na coluna 'sinal_ears_atend_lag2' preenchidos com a moda: 0
Valores NaN na coluna 'sinal_ears_atend_lag3' preenchidos com a moda: 0
Valores NaN na coluna 'sinal_evi_ivas_lag1' preenchidos com a moda: 0
Valores NaN na coluna 'sinal_evi_ivas_lag2' preenchidos com a moda: 0
Valores NaN na coluna 'sinal_evi_ivas_lag3' preenchidos com a moda: 0
Valores NaN na coluna 'EWS_ISF_lag1' preenchidos com a moda: 0
Valores NaN na coluna 'EWS_ISF_lag2' preenchidos com a moda: 0
Valores NaN na coluna 'EWS_ISF_lag3' preenchidos com a moda: 0
Valores NaN na coluna 'EWS_LOF_lag1' preenchidos com a moda: 0
Valores NaN na coluna 'EWS_LOF_lag2' preenchidos com a moda: 0
Valores NaN na coluna 'EWS_LOF_lag3' preenchidos com a moda: 0
Valores NaN na coluna 'EWS_OCSVM_lag1' preenchidos com a moda: 0
Valores NaN na coluna 'EWS_OCSVM_lag2' preenchidos com a moda: 0
Valores NaN na coluna 'EWS_OCSVM_lag3' preenchidos com a moda: 0
V

## --- 2.3 Configuração das features para o modelo ---

In [44]:
# Colunas dos modelos base que servirão de entrada para o modelo
df = df_com_lags #com antecipacao
base_models_columns=features_para_treino #configurado acima

#==============================================================
# variável alvo 
target_column = 'mem_surge_01_correct_with_consec'#'warning_final_mem_surge_01' 
#==============================================================


## --- 2.4 Divisao dos dados  ---

In [45]:
# X são as predições dos modelos base (features)
X = df[base_models_columns]
# y é o resultado verdadeiro que queremos prever
y = df[target_column]

In [46]:
# ---  Divisão em Dados de Treino e Teste ---
# Dividimos os dados para treinar o blender e depois testar sua eficácia
# Usamos 70% para treino e 30% para teste.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# --- 3. Emsemble MLP (MLPClassifier)  ---


In [47]:
# --- Treinamento do Blender (SMOTE + MLP) --- 
#Treinando o modelo Stacking com balanceameneto SMOTE (Synthetic Minority Over-sampling Technique) embutido via Pipeline para evitar vazamento (data leakage)

# 1. Construindo o "Tubo" (Pipeline) que executa os passos em ordem
mlp_blender_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)), # Passo 1: Cria os dados sintéticos
    ('mlp', MLPClassifier(             # Passo 2: Treina a rede neural
        hidden_layer_sizes=(10, 5), #camadas e qtd neuronios
        max_iter=1000, #epocas
        random_state=42 
    ))
])

# 2. Treinamos o pipeline com os dados ORIGINAIS (desbalanceados)
# O Pipeline vai aplicar o SMOTE apenas internamente, e passar o resultado pro MLP.
mlp_blender_pipeline.fit(X_train, y_train) 

print("Treinamento concluído.")

Treinamento concluído.


# --- 4 Avaliação do modelo e relatórios ---

In [48]:
# ==============================================================================
# 1. PREPARAÇÃO DOS DADOS (ORDENAÇÃO TEMPORAL)
# ==============================================================================

# Recria o dataframe com as informações necessárias
df_eval = df.loc[X_test.index].copy()
df_eval['y_true'] = y_test

# configuracao do modelo a ser avaliado 
#Binario = xgb_model
df_eval['y_prob'] = mlp_blender_pipeline.predict_proba(X_test)[:, 1]

# Ordena cronologicamente (ajuste 'co_ibge7' se necessário)
if 'co_ibge7' in df_eval.columns:
    df_eval = df_eval.sort_values(by=['co_ibge7', 'year_week'])
else:
    df_eval = df_eval.sort_values(by=['year_week'])

In [49]:
# ==============================================================================
# 2. FUNÇÃO DE CÁLCULO DE MÉTRICAS 
# ==============================================================================

def calcular_metricas_por_threshold_evento(df, threshold, window_weeks=3):
    # Aplica o corte
    y_pred = (df['y_prob'] >= threshold).astype(int)
    
    # --- A. Métricas de Classificação (Semana a Semana) ---
    tn, fp, fn, tp = confusion_matrix(df['y_true'], y_pred).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # MÉTRICAS DE MACHINE LEARNING:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1_score = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    
    
    # --- B. Métricas Epidemiológicas (Por Evento de Surto) ---
    # Identifica onde os surtos começam (transição 0 -> 1 no gabarito)
    y_true_series = pd.Series(df['y_true'].values)
    y_pred_series = pd.Series(y_pred.values)
    
    outbreak_starts = y_true_series[(y_true_series.diff() == 1) & (y_true_series == 1)].index
    total_outbreaks = len(outbreak_starts)
    
    early, timely, missed = 0, 0, 0
    
    for start in outbreak_starts:
        # Define janela de antecipação (ex: 3 semanas antes)
        w_start = max(0, start - window_weeks)
        
        # Verifica se houve alerta na janela anterior
        if y_pred_series[w_start:start].sum() > 0:
            early += 1
        # Verifica se houve alerta no dia
        elif y_pred_series[start] == 1:
            timely += 1
        # Se não houve nenhum dos dois
        else:
            missed += 1

    # --- C. Métricas de Falsos Positivos por Evento ---
    # Cria uma máscara onde é 1 apenas nas semanas que são FP (y_true=0 e y_pred=1)
    fp_mask = ((y_true_series == 0) & (y_pred_series == 1)).astype(int)
    
    # Conta quantas vezes a máscara passa de 0 para 1 (início de um evento de alarme falso)
    fp_events = (fp_mask.diff() == 1).sum()
    
    # Tratamento: se a primeira linha de todas do dataset já for um alarme falso, 
    # o diff() dá NaN, então adicionamos 1 manualmente para não perder essa contagem.
    if not fp_mask.empty and fp_mask.iloc[0] == 1:
        fp_events += 1
            
    return {
        "Threshold": threshold,
        # Contagens Epidemiológicas
        "Total Surtos (Eventos)": total_outbreaks,
        "Early Detection (Count)": early,
        "Timely Detection (Count)": timely,
        "Missed Outbreaks (Count)": missed,
        # Percentuais Epidemiológicos
        "Early Detection (%)": f"{early/total_outbreaks:.2%}" if total_outbreaks > 0 else "0.00%",
        "Timely Detection (%)": f"{timely/total_outbreaks:.2%}" if total_outbreaks > 0 else "0.00%",
        "Missed Outbreaks (%)": f"{missed/total_outbreaks:.2%}" if total_outbreaks > 0 else "0.00%",
        "Cobertura Total (%)": f"{(early+timely)/total_outbreaks:.2%}" if total_outbreaks > 0 else "0.00%",
        # Métricas de Machine Learning (comumente usadas em Computação)
        "Accuracy": f"{accuracy:.2%}",
        "Precision (PPV)": f"{precision:.2%}",
        "Sensitivity (Recall)": f"{sensitivity:.2%}",
        "Specificity": f"{specificity:.2%}",
        "F1-Score": f"{f1_score:.4f}", # se quiser alterar para % usar: f"{f1_score:.2%}"
        # Métricas de Classificação Clássicas
        "Sensitivity (Recall)": f"{sensitivity:.2%}",
        "Specificity": f"{specificity:.2%}",
        # Comparativo de Falsos Positivos (Semanas x Eventos)
        "Alarmes Falsos (Semanas/Linhas)": fp,
        "Eventos de Alarme Falso (FP Events)": int(fp_events)
    }

In [50]:
# ==============================================================================
# 3. GERAR RELATÓRIO COMPARATIVO
# ==============================================================================

# Calcula para os dois cenários
metrics_050 = calcular_metricas_por_threshold_evento(df_eval, 0.50)
#metrics_065 = calcular_metricas_por_threshold_evento(df_eval, 0.65)

# Cria DataFrame para exibição lado a lado
df_comparativo = pd.DataFrame([metrics_050]) # ,metrics_065
df_comparativo = df_comparativo.set_index("Threshold").T # Transpõe para ficar mais legível

print("=== RELATÓRIO MÉTRICAS ===")
print(df_comparativo)




=== RELATÓRIO MÉTRICAS ===
Threshold                               0.5
Total Surtos (Eventos)                29163
Early Detection (Count)               18027
Timely Detection (Count)               5540
Missed Outbreaks (Count)               5596
Early Detection (%)                  61.81%
Timely Detection (%)                 19.00%
Missed Outbreaks (%)                 19.19%
Cobertura Total (%)                  80.81%
Accuracy                             78.45%
Precision (PPV)                      42.16%
Sensitivity (Recall)                 63.52%
Specificity                          81.61%
F1-Score                             0.5068
Alarmes Falsos (Semanas/Linhas)       35934
Eventos de Alarme Falso (FP Events)   27218


# --- 4. Produção - inferência/aplicação do modelo ---

In [51]:
# ==============================================================================
# APLICAÇÃO DO MODELO EM PRODUÇÃO (artigo)
# Semana 2022-42 até o dado mais recente
# ==============================================================================

print("Preparando dados para inferência...")

# 1. DEFINIR O DATASET DE PRODUÇÃO
df_prod = df_com_lags[
    (df_com_lags['year_week'] >= '2022-42') & 
    (df_com_lags['year_week'] <= '2025-32')
].copy()

print(f"Dados filtrados: {len(df_prod)} semanas/registros encontrados a partir de 2022-42.")

Preparando dados para inferência...
Dados filtrados: 788655 semanas/registros encontrados a partir de 2022-42.


In [52]:
#=============================================================================
# rodar para o dado completo
#=============================================================================

#df_prod = df_com_lags

In [53]:
# 2. SEPARAR AS FEATURES DE ENTRADA DO  MODELO
# !!!!!!!!!EXATAMENTE a mesma e na MESMA ORDEM de treino do modelo

colunas_features = base_models_columns #configurado anteriormente acima
X_prod = df_prod[colunas_features]

In [54]:
# 3. VERIFICAÇÃO DE DADOS FALTANTES (Nulos)
# Se os dados não tiverem o sinal dos modelos de entrada, o modelo não pode prever.
if X_prod.isnull().values.any():
    print("Existem dados faltantes (NaN) nas features.")

In [55]:
# 4. FAZER AS PREDIÇÕES
print("Gerando probabilidades com o XGBoost...")
# Pega a probabilidade da classe 1 (Surto)
df_prod['prob_ensemble_mlp'] = mlp_blender_pipeline.predict_proba(X_prod)[:, 1]

Gerando probabilidades com o XGBoost...


In [56]:
# 5. APLICAR O LIMIAR DE ALERTA (THRESHOLDS)
# gerar alertas para 0.50 (definido em reunião para todos os modelos)
df_prod['signal_ensemble_mlp50'] = (df_prod['prob_ensemble_mlp'] >= 0.50).astype(int)

print("Predições concluídas com sucesso!")
print("-" * 60)

# 6. VISUALIZAR OS RESULTADOS MAIS RECENTES
colunas_visualizacao = ['year_week', 'prob_ensemble_mlp', 'signal_ensemble_mlp50'] 

display(df_prod[colunas_visualizacao].tail(10)) # Mostra as 10 últimas semanas preditas

Predições concluídas com sucesso!
------------------------------------------------------------


,year_week,prob_ensemble_mlp,signal_ensemble_mlp50
785079,2025-32,0.265969,0
785078,2025-32,0.265969,0
785077,2025-32,0.265969,0
785076,2025-32,0.265969,0
785075,2025-32,0.661462,1
785074,2025-32,0.278827,0
785073,2025-32,0.373577,0
785072,2025-32,0.329019,0
785070,2025-32,0.265969,0
788654,2025-32,0.265969,0


In [57]:
# ==============================================================================
# EXPORTANDO RESULTADOS DE PRODUÇÃO PARA PARQUET
# ==============================================================================
df_export = df_prod[['co_ibge',
 'epiweek',
 'year',
 'year_week',
 'week',
 'atend_ivas',
 'atend_totais',
 'mem_surge_01_correct_with_consec',
 'warning_final_mem_surge_01',
 'sinal_ears_atend',
 'sinal_evi_ivas',
 'EWS_ISF',
 'EWS_LOF',
 'EWS_OCSVM',
 'EWS_COPOD',
 'EWS_Rt',
 'prob_ensemble_xgb', 
 'signal_ensemble_xgb50',
 'prob_ensemble_mlp', 
 'signal_ensemble_mlp50']]


nome_arquivo_parquet = '/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/resultado_predicoes_producao_mlp.parquet'

# Salvando o dataframe
df_export.to_parquet(nome_arquivo_parquet, engine='pyarrow', index=False)
    
print("-" * 50)
print(f"Arquivo Parquet salvo com sucesso: {nome_arquivo_parquet}")
print(f"Total de linhas salvas: {len(df_export)}")
print("-" * 50)


--------------------------------------------------
Arquivo Parquet salvo com sucesso: /opt/storage/shared/aesop/aesop_shared/ensamble_modelling/resultado_predicoes_producao_mlp.parquet
Total de linhas salvas: 788655
--------------------------------------------------


# Verificar saída modelo

In [58]:
df_prod = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/resultado_predicoes_producao_mlp.parquet')
print(f"Tamanho do DataFrame: {df_prod.shape}")
print(f"Total de linhas: {len(df_prod)}")
print("\nVerificando dados faltantes (NaN):")
columns = ['prob_ensemble_mlp', 'signal_ensemble_mlp50']

for col in columns:
    print(f"Valores NaN na coluna '{col}' ")
    print(df_prod[col].isna().sum())

Tamanho do DataFrame: (788655, 20)
Total de linhas: 788655

Verificando dados faltantes (NaN):
Valores NaN na coluna 'prob_ensemble_mlp' 
0
Valores NaN na coluna 'signal_ensemble_mlp50' 
0


In [59]:
df_export.info()

<class 'pandas.core.frame.DataFrame'>
Index: 788655 entries, 0 to 788654
Data columns (total 20 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   co_ibge                           788655 non-null  int32  
 1   epiweek                           788655 non-null  int32  
 2   year                              788655 non-null  float64
 3   year_week                         788655 non-null  object 
 4   week                              788655 non-null  object 
 5   atend_ivas                        788655 non-null  int32  
 6   atend_totais                      788655 non-null  int32  
 7   mem_surge_01_correct_with_consec  788655 non-null  int32  
 8   warning_final_mem_surge_01        788655 non-null  int32  
 9   sinal_ears_atend                  788655 non-null  int64  
 10  sinal_evi_ivas                    788655 non-null  int64  
 11  EWS_ISF                           788655 non-null  int64 

In [60]:
# ==============================================================================
# AVALIAÇÃO FINAL NO DATASET DE PRODUÇÃO
# ==============================================================================

# 1. Prepare as Features (X) e o Gabarito (y) da produção
colunas_esperadas_pelo_modelo = mlp_blender_pipeline.feature_names_in_

X_prod = df_prod[colunas_esperadas_pelo_modelo].copy()

y_prod_true = df_prod['warning_final_mem_surge_01'] 

# 2. Gere as probabilidades com o modelo treinado
print("Gerando predições para o período de produção...")
probabilidades_prod = mlp_blender_pipeline.predict_proba(X_prod)[:, 1]

# 3. Cria um DataFrame temporário só com o que a função precisa ler
df_avaliacao_prod = pd.DataFrame({
    'y_true': y_prod_true.values,
    'y_prob': probabilidades_prod
})

# 4. Roda função de métricas
print(f"\n--- Resultados de Produção")
resultados_prod = calcular_metricas_por_threshold_evento(df_avaliacao_prod, threshold=0.50)

# 5. Imprime os resultados formatados
for metrica, valor in resultados_prod.items():
    print(f"{metrica.ljust(35)}: {valor}")

KeyError: "['sinal_ears_atend_lag1', 'sinal_ears_atend_lag2', 'sinal_ears_atend_lag3', 'sinal_evi_ivas_lag1', 'sinal_evi_ivas_lag2', 'sinal_evi_ivas_lag3', 'EWS_ISF_lag1', 'EWS_ISF_lag2', 'EWS_ISF_lag3', 'EWS_LOF_lag1', 'EWS_LOF_lag2', 'EWS_LOF_lag3', 'EWS_OCSVM_lag1', 'EWS_OCSVM_lag2', 'EWS_OCSVM_lag3', 'EWS_COPOD_lag1', 'EWS_COPOD_lag2', 'EWS_COPOD_lag3', 'EWS_Rt_lag1', 'EWS_Rt_lag2', 'EWS_Rt_lag3'] not in index"